In [1]:
import pandas as pd

In [2]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))

In [3]:
train_df = pd.read_sql("SELECT * FROM churn_train", engine)
test_df = pd.read_sql("SELECT * FROM churn_test", engine)

print("Train:", train_df.shape)
print("Test:", test_df.shape)

train_df.head()

Train: (11819, 25)
Test: (3877, 25)


,account_id,snapshot_date,industry,country,employee_count,plan,seats,mrr,user_count,admin_count,...,paid_amount_90d,open_amount_90d,uncollectible_amount_90d,ticket_count_90d,high_priority_tickets_90d,urgent_tickets_90d,avg_satisfaction_90d,avg_resolution_hours_90d,unresolved_tickets_90d,churn_within_90d
0,102347,2025-06-01,Software,Brazil,26,Growth,26,624.0,18,4,...,75141.18,11396.88,0.0,1,0,0,5.0,85.70,0,0
1,103319,2025-02-01,Software,Canada,15,Starter,7,84.0,7,3,...,0.00,0.00,0.0,0,0,0,NaN,NaN,0,0
2,103319,2025-03-01,Software,Canada,15,Starter,7,84.0,7,3,...,0.00,0.00,0.0,0,0,0,NaN,NaN,0,0
3,103319,2025-04-01,Software,Canada,15,Starter,7,84.0,7,3,...,0.00,0.00,0.0,1,0,0,4.0,24.30,0,0
4,103319,2025-05-01,Software,Canada,15,Starter,7,84.0,7,3,...,0.00,0.00,0.0,2,0,0,4.5,67.95,0,0


In [4]:
target = "churn_within_90d"

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (11819, 24)
y_train: (11819,)
X_test: (3877, 24)
y_test: (3877,)


In [5]:
X_train.dtypes

account_id                            int64
snapshot_date                datetime64[us]
industry                                str
country                                 str
employee_count                        int64
plan                                    str
seats                                 int64
mrr                                 float64
user_count                            int64
admin_count                           int64
member_count                          int64
viewer_count                          int64
invoice_count_90d                     int64
total_invoiced_90d                  float64
avg_invoice_amount_90d              float64
paid_amount_90d                     float64
open_amount_90d                     float64
uncollectible_amount_90d            float64
ticket_count_90d                      int64
high_priority_tickets_90d             int64
urgent_tickets_90d                    int64
avg_satisfaction_90d                float64
avg_resolution_hours_90d        

In [6]:
X_train["snapshot_date"].describe()

count                         11819
mean     2024-09-22 08:27:41.866486
min             2023-05-01 00:00:00
25%             2024-05-01 00:00:00
50%             2024-11-01 00:00:00
75%             2025-03-01 00:00:00
max             2025-06-01 00:00:00
Name: snapshot_date, dtype: object

In [7]:
target = "churn_within_90d"

drop_cols = [
    "account_id",
    "snapshot_date"
]

categorical_features = [
    "industry",
    "country",
    "plan"
]

numeric_features = [
    col for col in X_train.columns
    if col not in drop_cols + categorical_features
]

print("Categorical:", categorical_features)
print("Numeric:", numeric_features)

Categorical: ['industry', 'country', 'plan']
Numeric: ['employee_count', 'seats', 'mrr', 'user_count', 'admin_count', 'member_count', 'viewer_count', 'invoice_count_90d', 'total_invoiced_90d', 'avg_invoice_amount_90d', 'paid_amount_90d', 'open_amount_90d', 'uncollectible_amount_90d', 'ticket_count_90d', 'high_priority_tickets_90d', 'urgent_tickets_90d', 'avg_satisfaction_90d', 'avg_resolution_hours_90d', 'unresolved_tickets_90d']


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Columns we don't want the model to use
drop_cols = [
    "account_id",
    "snapshot_date"
]

# Categorical features
categorical_features = [
    "industry",
    "country",
    "plan"
]

# Everything else is numeric
numeric_features = [
    col
    for col in X_train.columns
    if col not in drop_cols + categorical_features
]

# Numeric preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both pipelines
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(
    X_train.drop(columns=drop_cols)
)

# Apply the learned preprocessing to test data
X_test_processed = preprocessor.transform(
    X_test.drop(columns=drop_cols)
)

print("Processed train:", X_train_processed.shape)
print("Processed test:", X_test_processed.shape)

Processed train: (11819, 40)
Processed test: (3877, 40)


In [9]:
from sklearn.linear_model import LogisticRegression

baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

baseline_model.fit(
    X_train_processed,
    y_train
)

print("Baseline Logistic Regression trained successfully.")

Baseline Logistic Regression trained successfully.


In [10]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

# Predictions
y_pred = baseline_model.predict(X_test_processed)

# Probability of churn
y_prob = baseline_model.predict_proba(X_test_processed)[:, 1]

# Confusion matrix
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Classification metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# ROC-AUC
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

# PR-AUC
print("PR-AUC:", average_precision_score(y_test, y_prob))

Confusion Matrix:
[[2937  908]
 [  21   11]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9929    0.7638    0.8634      3845
           1     0.0120    0.3438    0.0231        32

    accuracy                         0.7604      3877
   macro avg     0.5024    0.5538    0.4433      3877
weighted avg     0.9848    0.7604    0.8565      3877

ROC-AUC: 0.4789255526657997
PR-AUC: 0.011117125473769883


In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# Train
rf_model.fit(
    X_train_processed,
    y_train
)

# Predictions
y_pred_rf = rf_model.predict(X_test_processed)

# Churn probabilities
y_prob_rf = rf_model.predict_proba(X_test_processed)[:, 1]

# Evaluation
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf, digits=4))

print("ROC-AUC:", roc_auc_score(y_test, y_prob_rf))
print("PR-AUC:", average_precision_score(y_test, y_prob_rf))

Confusion Matrix:
[[3845    0]
 [  32    0]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9917    1.0000    0.9959      3845
           1     0.0000    0.0000    0.0000        32

    accuracy                         0.9917      3877
   macro avg     0.4959    0.5000    0.4979      3877
weighted avg     0.9836    0.9917    0.9876      3877

ROC-AUC: 0.5066807542262679
PR-AUC: 0.008649359724580644


/Users/falesh/Data Science/Wasted Effort/Customer Retention Intelligence/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/falesh/Data Science/Wasted Effort/Customer Retention Intelligence/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/falesh/Data Science/Wasted Effort/Customer Retention Intelligence/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in 

In [12]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

# --------------------------------------------------
# 1. Split training snapshots
# --------------------------------------------------

positive = train_df[train_df["churn_within_90d"] == 1]
negative = train_df[train_df["churn_within_90d"] == 0]

print("Positive snapshots:", len(positive))
print("Negative snapshots:", len(negative))


# --------------------------------------------------
# 2. Numeric feature analysis
# --------------------------------------------------

numeric_features = [
    "employee_count",
    "seats",
    "mrr",
    "user_count",
    "admin_count",
    "member_count",
    "viewer_count",
    "invoice_count_90d",
    "total_invoiced_90d",
    "avg_invoice_amount_90d",
    "paid_amount_90d",
    "open_amount_90d",
    "uncollectible_amount_90d",
    "ticket_count_90d",
    "high_priority_tickets_90d",
    "urgent_tickets_90d",
    "avg_satisfaction_90d",
    "avg_resolution_hours_90d",
    "unresolved_tickets_90d"
]

numeric_results = []

for feature in numeric_features:

    pos = positive[feature].dropna()
    neg = negative[feature].dropna()

    if len(pos) == 0 or len(neg) == 0:
        continue

    u_stat, p_value = mannwhitneyu(
        pos,
        neg,
        alternative="two-sided"
    )

    pos_median = pos.median()
    neg_median = neg.median()

    numeric_results.append({
        "feature": feature,
        "positive_median": pos_median,
        "negative_median": neg_median,
        "median_difference": pos_median - neg_median,
        "p_value": p_value
    })

numeric_results = pd.DataFrame(numeric_results)

numeric_results["abs_difference"] = (
    numeric_results["median_difference"].abs()
)

numeric_results = numeric_results.sort_values(
    "p_value"
)

print("\n=== NUMERIC FEATURES ===")
print(
    numeric_results[
        [
            "feature",
            "positive_median",
            "negative_median",
            "median_difference",
            "p_value"
        ]
    ].to_string(index=False)
)


# --------------------------------------------------
# 3. Categorical feature analysis
# --------------------------------------------------

categorical_features = [
    "industry",
    "country",
    "plan"
]

print("\n=== CATEGORICAL FEATURES ===")

for feature in categorical_features:

    result = (
        train_df
        .groupby(feature)["churn_within_90d"]
        .agg(
            snapshots="count",
            churned="sum",
            churn_rate="mean"
        )
        .sort_values("churn_rate", ascending=False)
    )

    result["churn_rate"] = result["churn_rate"] * 100

    print(f"\n--- {feature} ---")
    print(result.to_string())


# --------------------------------------------------
# 4. Simple overall signal summary
# --------------------------------------------------

significant = numeric_results[
    numeric_results["p_value"] < 0.05
]

print("\n=== SIGNAL SUMMARY ===")
print(
    f"Numeric features with p < 0.05: "
    f"{len(significant)} / {len(numeric_results)}"
)

if len(significant) > 0:
    print("\nPotentially informative numeric features:")
    print(
        significant[
            ["feature", "median_difference", "p_value"]
        ].to_string(index=False)
    )
else:
    print(
        "No numeric feature shows a statistically significant "
        "difference at alpha = 0.05."
    )

Positive snapshots: 165
Negative snapshots: 11654

=== NUMERIC FEATURES ===
                  feature  positive_median  negative_median  median_difference  p_value
         ticket_count_90d              0.0             0.00               0.00 0.013074
       total_invoiced_90d              0.0             0.00               0.00 0.043272
             member_count              5.0             6.00              -1.00 0.044920
 uncollectible_amount_90d              0.0             0.00               0.00 0.071573
          paid_amount_90d              0.0             0.00               0.00 0.074464
        invoice_count_90d              0.0             0.00               0.00 0.080819
             viewer_count              2.0             2.00               0.00 0.088091
          open_amount_90d              0.0             0.00               0.00 0.100261
   avg_invoice_amount_90d             91.7           196.11            -104.41 0.119210
               user_count              7.0  

In [13]:
# ============================================
# FEASIBILITY CHECK — BEHAVIOURAL CHANGE
# ============================================

from scipy.stats import mannwhitneyu
import pandas as pd
import numpy as np

df = train_df.copy()

# Sort so each customer's previous snapshot is available
df = df.sort_values(["account_id", "snapshot_date"])

# Features where change may contain signal
change_features = [
    "mrr",
    "seats",
    "user_count",
    "member_count",
    "ticket_count_90d",
    "high_priority_tickets_90d",
    "urgent_tickets_90d",
    "unresolved_tickets_90d",
    "avg_satisfaction_90d",
    "avg_resolution_hours_90d",
    "total_invoiced_90d",
]

# Calculate change from previous snapshot
for feature in change_features:
    df[f"{feature}_change"] = (
        df.groupby("account_id")[feature].diff()
    )

# Only snapshots where a previous observation exists
change_df = df.dropna(
    subset=[f"{feature}_change" for feature in change_features],
    how="all"
)

positive = change_df[
    change_df["churn_within_90d"] == 1
]

negative = change_df[
    change_df["churn_within_90d"] == 0
]

results = []

for feature in change_features:

    change_col = f"{feature}_change"

    pos = positive[change_col].dropna()
    neg = negative[change_col].dropna()

    if len(pos) == 0 or len(neg) == 0:
        continue

    u_stat, p_value = mannwhitneyu(
        pos,
        neg,
        alternative="two-sided"
    )

    results.append({
        "feature": change_col,
        "churn_median_change": pos.median(),
        "non_churn_median_change": neg.median(),
        "median_difference": pos.median() - neg.median(),
        "p_value": p_value
    })

results = pd.DataFrame(results).sort_values("p_value")

print("=== BEHAVIOURAL CHANGE SIGNAL ===\n")

print(
    results.to_string(index=False)
)

print("\n=== FEASIBILITY ===")

significant = results[results["p_value"] < 0.05]

print(
    f"Significant change features: "
    f"{len(significant)} / {len(results)}"
)

if len(significant) > 0:
    print("\nPotential signal exists in:")
    print(
        significant[
            [
                "feature",
                "churn_median_change",
                "non_churn_median_change",
                "p_value"
            ]
        ].to_string(index=False)
    )
else:
    print(
        "\nNo meaningful statistical signal found "
        "in behavioural changes."
    )

=== BEHAVIOURAL CHANGE SIGNAL ===

                         feature  churn_median_change  non_churn_median_change  median_difference  p_value
         ticket_count_90d_change                  0.0                      0.0                0.0 0.022782
high_priority_tickets_90d_change                  0.0                      0.0                0.0 0.157419
       total_invoiced_90d_change                  0.0                      0.0                0.0 0.164926
       urgent_tickets_90d_change                  0.0                      0.0                0.0 0.206079
 avg_resolution_hours_90d_change                  0.0                      0.0                0.0 0.439028
   unresolved_tickets_90d_change                  0.0                      0.0                0.0 0.509456
     avg_satisfaction_90d_change                  0.0                      0.0                0.0 0.804594
                      mrr_change                  0.0                      0.0                0.0      NaN
  